In [34]:
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data, feature_names

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

In [35]:
# Set random seed for reproducibility across runs
# This ensures that your results will be the same every time you run the notebook
seed_everything(42)  # for reproducibility

In [36]:
import glob
# -----------------------------
# 1. LOAD ALL TRAINING CSVs
# -----------------------------
train_path = "../../datasets_challenge/train"

train_dfs = []
for f in sorted(glob.glob(os.path.join(train_path, "*.csv"))):
    print("Loaded:", f)
    train_dfs.append(pd.read_csv(f))

# Merge everything into one dataframe
train_df = pd.concat(train_dfs, ignore_index=True)
print("Total samples:", len(train_df))

Loaded: ../../datasets_challenge/train\train_part_001.csv
Loaded: ../../datasets_challenge/train\train_part_002.csv
Loaded: ../../datasets_challenge/train\train_part_003.csv
Loaded: ../../datasets_challenge/train\train_part_004.csv
Loaded: ../../datasets_challenge/train\train_part_005.csv
Loaded: ../../datasets_challenge/train\train_part_006.csv
Loaded: ../../datasets_challenge/train\train_part_007.csv
Loaded: ../../datasets_challenge/train\train_part_008.csv
Loaded: ../../datasets_challenge/train\train_part_009.csv
Loaded: ../../datasets_challenge/train\train_part_010.csv
Loaded: ../../datasets_challenge/train\train_part_011.csv
Loaded: ../../datasets_challenge/train\train_part_012.csv
Loaded: ../../datasets_challenge/train\train_part_013.csv
Loaded: ../../datasets_challenge/train\train_part_014.csv
Loaded: ../../datasets_challenge/train\train_part_015.csv
Loaded: ../../datasets_challenge/train\train_part_016.csv
Loaded: ../../datasets_challenge/train\train_part_017.csv
Loaded: ../../

In [37]:

# -----------------------------
# 2. LOAD SIDESLIP DISTRIBUTION
# -----------------------------
dist_df = pd.read_csv("../../datasets_challenge/sideSlip_distribution.csv")

# few-shot bins
few_bins = dist_df[dist_df["band"] == "few"]["sideSlip"].tolist()
few_bins = [round(x, 2) for x in few_bins]

print("Few-shot bins:", few_bins[:20], " ...")
print("Number of few-shot bins:", len(few_bins))

Few-shot bins: [-14.9, -14.2, -14.0, -13.7, -12.7, -12.1, -12.0, -11.9, -11.7, -11.6, -11.5, -11.4, -11.3, -11.2, -11.1, -11.0, -10.9, -10.8, -10.7, -10.6]  ...
Number of few-shot bins: 151


In [38]:
# -----------------------------
# 3. FILTER INTO SPLITS
# -----------------------------

# Round sideSlip so it matches distribution CSV bin values
train_df["sideSlip_rounded"] = train_df["sideSlip"].round(3)

# A) Full dataset (nothing filtered)
df_A = train_df.copy()

# B) Tail dataset = ONLY rows falling in few-shot bins
df_B = train_df[ train_df["sideSlip_rounded"].isin(few_bins) ].copy()

# C) Curriculum dataset:
#    - Include ALL tail rows
#    - Include sampled many-shot rows (to maintain balance)

# Extract many-shot bins
many_bins = dist_df[dist_df["band"] == "many"]["sideSlip"].tolist()
many_bins = [round(x, 2) for x in many_bins]

df_many = train_df[train_df["sideSlip_rounded"].isin(many_bins)].copy()

# Sample many-shot at fraction (you can tune this)
SAMPLE_FRACTION = 0.10  # 10% of many-shot to mix with tail

df_many_sampled = df_many.sample(frac=SAMPLE_FRACTION, random_state=42)

# Curriculum = all few-shot + small many-shot sample
df_C = pd.concat([df_B, df_many_sampled], ignore_index=True)

print("\nDataset sizes:")
print("Model A (full):", len(df_A))
print("Model B (tail only):", len(df_B))
print("Model C (curriculum):", len(df_C))



Dataset sizes:
Model A (full): 275226
Model B (tail only): 422
Model C (curriculum): 14983


In [39]:
# Output folders
os.makedirs("data_model_A", exist_ok=True)
os.makedirs("data_model_B", exist_ok=True)
os.makedirs("data_model_C", exist_ok=True)

# Save as CSVs
df_A.to_csv("data_model_A/train.csv", index=False)
df_B.to_csv("data_model_B/train.csv", index=False)
df_C.to_csv("data_model_C/train.csv", index=False)

print("Saved dataset splits.")


Saved dataset splits.


In [40]:
train_loader_A, val_loader_A, pipe_A = load_data(
    train_path="data_model_A",
    train_ratio=0.98,
    batchsize=64,
    columns_to_drop=["sideSlip","vyCG","time","longitude","latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

train_loader_B, val_loader_B, pipe_B = load_data(
    train_path="data_model_B",
    train_ratio=None,  # no need for validation; train ONLY tail
    batchsize=64,
    columns_to_drop=["sideSlip","vyCG","time","longitude","latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

train_loader_C, val_loader_C, pipe_C = load_data(
    train_path="data_model_C",
    train_ratio=0.95,  # curriculum should have validation
    batchsize=64,
    columns_to_drop=["sideSlip","vyCG","time","longitude","latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

print("All dataloaders ready.")


data_model_A\train.csv
data_model_B\train.csv
data_model_C\train.csv
All dataloaders ready.


In [41]:
for X, y in train_loader_A:
    print(X.shape)
    break


torch.Size([64, 20, 23])


In [42]:
class RNN(nn.Module):
    """
    Recurrent Neural Network for sideslip angle prediction.
    
    Architecture:
    1. RNN Layer - Processes sequential input
    2. Fully Connected Layer - Transforms RNN output
    3. Output Layer - Produces final prediction
    
    Dropout layers are included to prevent overfitting.
    """
    def __init__(self, input_size, output_size=1, rnn_units=28, fc_units=28, 
                 activation_fct=nn.ReLU(), dropout=[0.0, 0, 0], rnn_nonlinearity='relu'):
        super(RNN, self).__init__()
        
        # RNN layer: processes the sequence and captures temporal dependencies
        self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                num_layers=1, nonlinearity=rnn_nonlinearity, batch_first=True)
        
        # Fully connected layers for processing RNN output
        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, output_size)
        
        # Dropout layers to prevent overfitting (randomly zeros out neurons during training)
        self.dropout1 = nn.Dropout(dropout[0])  # Applied to input
        self.dropout2 = nn.Dropout(dropout[1])  # Applied after RNN
        self.dropout3 = nn.Dropout(dropout[2])  # Applied after FC layer
        
        self.activation_fct = activation_fct
 
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        x = self.dropout1(x)
        
        # Process sequence through RNN
        x, _ = self.rnn_layer(x)  # Output shape: (batch_size, sequence_length, rnn_units)
        
        # Take only the last time step's output (t0)
        x = x[:, -1, :]  # Shape: (batch_size, rnn_units)
        
        x = self.dropout2(x)
        x = self.activation_fct(self.fc(x))  # Apply fully connected layer with activation
        x = self.dropout3(x)
        x = self.output_layer(x)  # Final prediction
        
        return x

In [43]:
# Set device for computation (CPU or GPU)
device = "cpu"  # Change to "cuda" if you have a GPU available

# Model hyperparameters
input_size = 22 # Number of features (22 in this case)
rnn_units = 48  # Number of units in RNN hidden state
fc_units = 48  # Number of units in fully connected layer
output_size = 1  # Single output (sideslip angle)
lr = 0.0003  # Learning rate for optimizer

# Initialize the model with specified architecture
model = RNN(
    input_size=input_size, 
    output_size=output_size,
    rnn_units=rnn_units,
    fc_units=fc_units,
    dropout=[0.0, 0.0, 0.0]  # Dropout rates for [input, after_rnn, after_fc]
).to(device)

# Define loss function and optimizer
criterion = nn.L1Loss()  # Mean Absolute Error (MAE) - measures average prediction error
optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # Adam optimizer for weight updates

In [44]:
# make it a method so i can call it multiple times
def train_model(model, train_dataloader, val_dataloader=None, num_epochs=60, early_stop_thresh=10, device="cpu", model_name="model"):
    # Training configuration
    best_loss = np.inf  # Initialize with infinity
    best_epoch = -1
    best_model_state = None  # Will store the best model weights

    # Lists to track training progress
    train_curve = []
    val_curve = []

    for epoch in range(num_epochs):
        running_loss = 0
        
        # ========== TRAINING PHASE ==========
        model.train()  # Set model to training mode (enables dropout)
        for step, (x, y) in enumerate(train_dataloader):
            # Move data to device and reshape target
            X, y = x.to(device), y.to(device).reshape([len(y), 1])
            
            optimizer.zero_grad()  # Reset gradients from previous iteration
            pred = model(X)  # Forward pass: get predictions
            train_loss = criterion(pred, y)  # Calculate loss
            running_loss += train_loss.item()
            train_loss.backward()  # Backward pass: compute gradients
            optimizer.step()  # Update model weights
        
        avg_train_loss = running_loss / step
        train_curve.append(avg_train_loss)

        if val_dataloader is not None:
            # ===== VALIDATION =====
            model.eval()
            running_val_loss = 0
            with torch.no_grad():
                for val_step, (x, y) in enumerate(val_dataloader):
                    X_val, y_val = x.to(device), y.to(device).reshape([len(y), 1])
                    val_loss = criterion(model(X_val), y_val)
                    running_val_loss += val_loss.item()
            avg_val_loss = running_val_loss / (val_step + 1)
            val_curve.append(avg_val_loss)

            print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}", end="")
            
            # EARLY STOPPING BASED ON VALIDATION
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                best_epoch = epoch
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "loss": best_loss
                }, f"{model_name}_best.pth")

                print(f" → New best validation loss!")
            elif epoch - best_epoch > early_stop_thresh:
                print(" → Early stopping!")
                break
            else:
                print("")
        else:
            # ===== NO VALIDATION MODE (TRAIN ON 100%) =====
            print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.5f}")


    # ========== LOAD BEST MODEL ==========
    if best_epoch != -1:
        checkpoint = torch.load(f"{model_name}_best.pth", map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])

        print(f"\n✓ Best model from epoch {best_epoch+1} loaded (Val Loss: {best_loss:.5f})")
    else:
        print("\n⚠ No improvement found, using current model")

In [45]:
input_size = train_loader_A.dataset[0][0].shape[1]
print("Actual input size =", input_size)
device = "cpu"

# -----------------------------------------------
# MODEL A — Base generalist model
# -----------------------------------------------
model_A = RNN(input_size=input_size, rnn_units=48, fc_units=48).to(device)
train_model(model_A, train_loader_A, val_loader_A, model_name="model_A")

# -----------------------------------------------
# MODEL B — Tail/few-shot specialist
# -----------------------------------------------
model_B = RNN(input_size=input_size, rnn_units=48, fc_units=48).to(device)
train_model(model_B, train_loader_B, val_loader=None, model_name="model_B")

# -----------------------------------------------
# MODEL C — Curriculum model
# -----------------------------------------------
model_C = RNN(input_size=input_size, rnn_units=48, fc_units=48).to(device)
train_model(model_C, train_loader_C, val_loader_C, model_name="model_C")


Actual input size = 23
Epoch 1/60 | Train Loss: 0.88172 | Val Loss: 0.87720 → New best validation loss!
Epoch 2/60 | Train Loss: 0.88168 | Val Loss: 0.87719 → New best validation loss!
Epoch 3/60 | Train Loss: 0.88168 | Val Loss: 0.87698 → New best validation loss!
Epoch 4/60 | Train Loss: 0.88170 | Val Loss: 0.87634 → New best validation loss!
Epoch 5/60 | Train Loss: 0.88165 | Val Loss: 0.87703
Epoch 6/60 | Train Loss: 0.88165 | Val Loss: 0.87665
Epoch 7/60 | Train Loss: 0.88173 | Val Loss: 0.87742
Epoch 8/60 | Train Loss: 0.88168 | Val Loss: 0.87683
Epoch 9/60 | Train Loss: 0.88166 | Val Loss: 0.87643
Epoch 10/60 | Train Loss: 0.88166 | Val Loss: 0.87605 → New best validation loss!
Epoch 11/60 | Train Loss: 0.88171 | Val Loss: 0.87608
Epoch 12/60 | Train Loss: 0.88164 | Val Loss: 0.87717
Epoch 13/60 | Train Loss: 0.88170 | Val Loss: 0.87635
Epoch 14/60 | Train Loss: 0.88170 | Val Loss: 0.87669
Epoch 15/60 | Train Loss: 0.88165 | Val Loss: 0.87810
Epoch 16/60 | Train Loss: 0.88174 | 

TypeError: train_model() got an unexpected keyword argument 'val_loader'

In [ ]:
# Load test dataloader
test_dataloader = load_test_data(
    test_path="../../datasets_challenge/test",
    batchsize=64,
    pipeline=pipe_A,
    load_anonymized_data=True
)

# ------------------------------------
# Load trained weights
# ------------------------------------
model_A.load_state_dict(torch.load("model_A_best.pth")["model_state_dict"])
model_B.load_state_dict(torch.load("model_B_best.pth")["model_state_dict"])
model_C.load_state_dict(torch.load("model_C_best.pth")["model_state_dict"])

model_A.eval()
model_B.eval()
model_C.eval()

preds_A = []
preds_B = []
preds_C = []

device = "cpu"

with torch.no_grad():
    for X, _ in test_dataloader:
        X = X.to(device)

        preds_A.append(model_A(X).cpu().numpy())
        preds_B.append(model_B(X).cpu().numpy())
        preds_C.append(model_C(X).cpu().numpy())

preds_A = np.concatenate(preds_A).reshape(-1)
preds_B = np.concatenate(preds_B).reshape(-1)
preds_C = np.concatenate(preds_C).reshape(-1)


../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv


In [ ]:
final_pred = 0.5 * preds_A + 0.3 * preds_B + 0.2 * preds_C

In [ ]:
submission = pd.DataFrame({
    "predicted_sideslip": final_pred,
    "id": range(len(final_pred))
})

submission.to_csv("predictions_staged_ensemble.csv", index=False)
print("✓ Saved predictions_staged_ensemble.csv")


✓ Saved predictions_staged_ensemble.csv
